In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import os
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None
from IPython.display import display

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
file_path = os.path.join("..", "data", "processed", "feature_select.csv")

# Read CSV with index_col=0 to avoid unnamed column issues
data = pd.read_csv(file_path, index_col=0)
data

In [ ]:
data.columns

In [ ]:
print(data['Exit_Status'].value_counts())

In [ ]:
# Define a mapping for Exit_Status
exit_status_mapping = {
    'graduating': 0,
    'switching': 1,
    'dropping': 2
}

# Apply the mapping to replace categorical values with numbers
data['Exit_Status'] = data['Exit_Status'].map(exit_status_mapping)

# Show the first few rows to verify
print(data['Exit_Status'].value_counts())


In [ ]:
print(data['Exit_Status'].value_counts())

In [ ]:
# From the last result from the EDA.ipynb, we saw that Prior_Edu_Postcode, Prior_Edu_Country and Study_Prog_Exam_Completed have a corr >0.7, therefore we can drop them here.
# However, we are dropping them from the already one-hot-encoded data called feature_select

# Drop columns that start with 'Prior_Edu_Postcode' or 'Prior_Edu_Country'
cols_to_drop = [col for col in data.columns if col.startswith("Prior_Edu_Postcode") or col.startswith("Prior_Edu_Country")]

# Print columns to be removed
print("Columns to be removed:")
print(cols_to_drop)

# Drop the columns
data = data.drop(columns=cols_to_drop)

# Verify the remaining columns
print("Remaining columns:")
print(data.columns)



Print only the pairs of columns with correlation above 0.7

In [ ]:
# Convert Exit_Status to numeric values automatically (if not already done)
data['Exit_Status'] = data['Exit_Status'].astype('category').cat.codes  # This will assign 0,1,2 automatically

# Drop Enrollment_ID (since it's just an identifier)
data_numeric = data.drop(columns=['Enrollment_ID'])

# Compute correlation matrix
corr_matrix = data_numeric.corr()

# Find pairs with correlation above 0.7 (excluding self-correlation)
high_corr_pairs = []
threshold = 0.7

for col1 in corr_matrix.columns:
    for col2 in corr_matrix.columns:
        if col1 != col2 and corr_matrix.loc[col1, col2] > threshold:
            high_corr_pairs.append((col1, col2, corr_matrix.loc[col1, col2]))

# Remove duplicate pairs (since correlation is symmetric)
unique_high_corr_pairs = set(tuple(sorted(pair[:2])) + (pair[2],) for pair in high_corr_pairs)

# Print highly correlated column pairs
print("Highly Correlated Column Pairs (Above 70%):")
for col1, col2, corr in unique_high_corr_pairs:
    print(f"{col1} - {col2}: {corr:.2f}")


In [ ]:
del data['Study_Prog_Exam_Completed']

In [ ]:
data_final.to_csv(os.path.join("..", "data", "processed", "removed_high_corr.csv"), header=True)

In [ ]:
data = data_final
data

In [ ]:
print(data['Exit_Status'].value_counts())

# Run a stats test to check for significance

In [ ]:
data.columns

In [ ]:
# Drop Enrollment_ID if present
if 'Enrollment_ID' in data.columns:
    data = data.drop(columns=['Enrollment_ID'])

# Ensure Exit_Status is numeric (already mapped previously)
y = data['Exit_Status']

# Use all other columns as predictors (X)
X = data.drop(columns=['Exit_Status'])

# Add constant term for intercept
X = sm.add_constant(X)

# Fit OLS model
model = sm.OLS(y, X).fit()

# Print summary
print(model.summary())

# Extract statistically significant features (p-value < 0.05)
significant_features = model.pvalues[model.pvalues < 0.05].index
print("\nSignificant Features (p < 0.05):")
print(significant_features)

# Extract non-significant features (p-value >= 0.05)
non_significant_features = model.pvalues[model.pvalues >= 0.05].index
print("\nNon-Significant Features (p >= 0.05):")
print(non_significant_features)


Highly statisfically significant features on Exiting include: 
Study_Prog_Exam_Completed
Study_Prog_Group_Size
Student_Enrollment_Gap
Study_Prog_Name( 19 features from it)
Prior_Edu_School_Location (22 features from it)

while the non significant features on Exiting include: 
Prior_Edu_Type
Study_Prog_Name
Prior_Edu_School_Location (more than 50 features from it)

Ambigious features: 
Study_Prog_Name
Prior_Edu_School_Location  

In [ ]:
# Set options to display all rows and columns
pd.set_option('display.max_rows', None)        # Show all rows
pd.set_option('display.max_columns', None)     # Show all columns
pd.set_option('display.max_colwidth', None)    # Show full content of each column
pd.set_option('display.width', None)           # Adjust width for better readability
pd.set_option('display.expand_frame_repr', True)  # Expand the DataFrame across multiple lines if necessary
